# Twin-2K-500 data exploration (Deliverable 1)

This notebook is a **data exploration report**, not a modeling notebook. It inspects the local Twin-2K-500 files, reconstructs the intended hold-out scoring protocol, and computes human test–retest reliability from T1/T2 answer blocks.

Run **from the repository root** with `data/` already populated:

```bash
jupyter notebook notebooks/01_data_exploration.ipynb
# or: poetry run jupyter nbconvert --to notebook --execute notebooks/01_data_exploration.ipynb
```

Reusable helpers live in `src/data_utils.py` and `src/evaluation_utils.py`. Raw files under `data/` are never modified.

---

## 1. Executive summary

Findings below were **computed from the local files** (see later sections for methods). They are not copied from the paper.

- **N = 2,058** participants. Participant IDs `1…2058` are identical across persona JSON, T1 blocks, T2 blocks, persona summaries, and all four wave CSVs.
- **Two Hugging Face views, four local representations:** `wave_split` supplies the persona JSON plus T1/T2 hold-out blocks; `full_persona` supplies prose summaries. Raw Qualtrics CSVs are a fifth, rectangular export.
- **Natural prediction setup:** condition on waves 1–3 *non-holdout* answers (persona); predict the same person’s answers on **repeated** hold-out tasks. Wave 4 is a **retest of the same tasks/conditions**, not an independent new test set.
- **Human test–retest (headline):** equal-task-weight accuracy **81.73%** (mean of 17 task means; person-level mean is the same because every completer has all 17 tasks). 95% CI across people: **81.42–82.04%**. Paper reports 81.72% — a **0.01pp** difference, treated as a rounding/validation check, not as our source.
- **Task reliability is not uniform:** about **72%** (denominator neglect) to **89%** (WTA/WTP). Person-level accuracy IQR is roughly **77–87%**.
- **Sample:** US adults recruited on Prolific with age/sex/ethnicity quotas; this release is **four-wave completers only**. Attrition from the original Wave 1 N cannot be seen in these files. Coarse demographics; not a proof of full US representativeness.


In [1]:
from pathlib import Path
import os, sys, json, warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
np.random.seed(42)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 120)

ROOT = Path.cwd()
if not (ROOT / "data" / "mega_persona_json").exists():
    if (ROOT.parent / "data" / "mega_persona_json").exists():
        ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_utils import (
    DATA_DIR, PERSONA_DIR, ANSWER_BLOCK_DIR, SUMMARY_DIR, WAVE_CSV_DIR, REPO_ROOT,
    count_local_files, collect_pid_sets, load_wave, load_wave_csv, inspect_csv_header,
    load_demographics, DEMOGRAPHIC_COLUMNS, load_json, iter_questions, iter_elements,
    summarize_persona_files, summarize_answer_block_files, compare_all_t1_t2_structure,
    find_example_question, sanitize_question_example, pid_from_filename, item_level_response_count,
)
from src.evaluation_utils import (
    RNG_SEED, get_column_ranges, get_qid_to_task, task_catalog, item_accuracy,
    extract_numeric_answers, load_importid_mapping, answers_to_eval_columns,
    load_holdout_matrices, apply_anchoring_deciles, compute_test_retest,
    assign_decile, ANCHOR_GROUP_A, ANCHOR_GROUP_B,
)

assert ROOT == REPO_ROOT or ROOT.resolve() == REPO_ROOT.resolve()
FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "regular",
    "font.size": 11,
})
print("Repository root:", ROOT)
print("Random seed:", RNG_SEED)


Repository root: /home/user/PycharmProjects/Twin-2K-500-Assignment/Digital-Twin-Simulation
Random seed: 42


## 2. Dataset structure

The Hugging Face dataset has two configs. `download_dataset.py` writes them to disk as follows.

**`wave_split`** (one row per person)

| Field | Local file | Role |
|---|---|---|
| `pid` | filename `pid_{id}_…` | Participant id (`TWIN_ID` in CSVs) |
| `wave1_3_persona_json` | `data/mega_persona_json/mega_persona/` | Waves 1–3 **non-holdout** Q&A → **persona / model input** |
| `wave4_Q_wave1_3_A` | `…/answer_blocks/pid_*_wave4_Q_wave1_3_A.json` | Hold-out **task shells + original answers** → **T1** |
| `wave4_Q_wave4_A` | `…/answer_blocks/pid_*_wave4_Q_wave4_A.json` | Same shells + Wave 4 answers → **T2** |

**`full_persona`**

| Field | Local file | Role |
|---|---|---|
| `pid` | filename | Same ids |
| `persona_summary` | `data/mega_persona_summary_text/` | Prose sketch of the person (optional encoding) |

T1/T2 terminology used in this report:

- **T1** = first administration of a hold-out task (during waves 1–3).
- **T2** = the **same task and condition** repeated in Wave 4.
- T1 vs T2 is **human test–retest**, not train vs a disjoint test set.

```
Waves 1–3 non-holdout responses  →  persona / model input
                                    (must not include T1 hold-out answers)

Held-out task answers at original administration  →  T1
Same tasks, same conditions, repeated in Wave 4     →  T2

T1 vs T2  →  human test–retest reliability
             (practical ceiling used by the paper; not a mathematical upper bound)
```


In [2]:
file_table = count_local_files()
display(file_table)

print("Raw CSV files:")
for p in sorted(WAVE_CSV_DIR.glob("*.csv")):
    print(f"  {p.name:40s} {p.stat().st_size/1e6:5.1f} MB")

pid_sets = collect_pid_sets()
print("\nPID coverage")
for name, s in pid_sets.items():
    s = {x for x in s if x is not None}
    print(f"  {name:10s} n={len(s):4d}  min={min(s)}  max={max(s)}")

base = pid_sets["persona"]
for name, s in pid_sets.items():
    print(f"  {name} symmetric difference vs persona: {s ^ base}")

csv_ids = set(load_wave(1, "labels")["TWIN_ID"].dropna().astype(int))
print("Wave-1 CSV vs persona:", csv_ids ^ base)
print("IDs are consecutive 1..2058:", base == set(range(1, 2059)))


,representation,participant_count,role,local_location
0,wave1_3_persona_json (persona),2058,Waves 1–3 non-holdout responses; model/persona input,data/mega_persona_json/mega_persona
1,wave4_Q_wave1_3_A (T1),2058,Hold-out tasks with original (waves 1–3) answers,data/mega_persona_json/answer_blocks
2,wave4_Q_wave4_A (T2),2058,Same hold-out tasks with Wave 4 repeated answers,data/mega_persona_json/answer_blocks
3,persona_summary (full_persona),2058,Prose summary of the person; optional persona encoding,data/mega_persona_summary_text
4,raw Qualtrics wave CSVs,see per-file row counts,Original survey exports (labels + numeric codes),data/wave_csv


Raw CSV files:
  wave_1_labels_anonymized.csv              12.1 MB
  wave_1_numbers_anonymized.csv              5.8 MB
  wave_2_labels_anonymized.csv               9.8 MB
  wave_2_numbers_anonymized.csv              7.3 MB
  wave_3_labels_anonymized.csv              14.1 MB
  wave_3_numbers_anonymized.csv              6.6 MB
  wave_4_labels_anonymized.csv              11.5 MB
  wave_4_numbers_anonymized.csv              7.1 MB

PID coverage
  persona    n=2058  min=1  max=2058
  t1         n=2058  min=1  max=2058
  t2         n=2058  min=1  max=2058
  summary    n=2058  min=1  max=2058
  persona symmetric difference vs persona: set()
  t1 symmetric difference vs persona: set()
  t2 symmetric difference vs persona: set()
  summary symmetric difference vs persona: set()
Wave-1 CSV vs persona: set()
IDs are consecutive 1..2058: True


## 3. Raw data / schema inspection

Each `*_anonymized.csv` is a **Qualtrics export**, not a clean one-row header table:

1. Row 0 — variable names (`TWIN_ID`, `Q12`, `Big Five _1`, …)
2. Row 1 — human-readable question labels
3. Row 2 — `{"ImportId": "QID…"}` JSON (maps CSV columns to survey QID)
4. Row 3+ — participants

`src.data_utils.load_wave_csv` skips rows 1–2. **Labels** files store category text; **numbers** files store the same columns as codes (e.g. Male/Female → 1/2). Codes are **ordinal/categorical positions**, not continuous measurements, unless the item is numeric (slider, text-entry number).


In [3]:
header_rows = []
shapes = []
for wave in range(1, 5):
    for kind in ("labels", "numbers"):
        path = WAVE_CSV_DIR / f"wave_{wave}_{kind}_anonymized.csv"
        info = inspect_csv_header(path)
        header_rows.append({"file": info["file"], "n_columns": info["n_columns"],
                            "row0": info["row0_first"], "importid_counts": info["importid_counts"]})
        df = load_wave(wave, kind)
        shapes.append({
            "file": path.name,
            "n_rows": len(df),
            "n_cols": df.shape[1],
            "duplicate_TWIN_ID": int(df["TWIN_ID"].duplicated().sum()),
            "missing_TWIN_ID": int(df["TWIN_ID"].isna().sum()),
            "Finished=TRUE": int((df["Finished"].astype(str).str.upper() == "TRUE").sum()) if "Finished" in df.columns else np.nan,
        })

print("First rows of each CSV (row0 = names, row2 ≈ ImportId):")
display(pd.DataFrame(header_rows))
print("Clean-loaded participant tables:")
display(pd.DataFrame(shapes))

w1_lab, w1_num = load_wave(1, "labels"), load_wave(1, "numbers")
print("Labels vs numbers share columns:", list(w1_lab.columns) == list(w1_num.columns))
print("\nSex (Q12): labels vs numeric codes")
print(pd.crosstab(w1_lab["Q12"], w1_num["Q12"]))
print("\nBig Five item 1: labels vs codes")
print(pd.crosstab(w1_lab["Big Five _1"], w1_num["Big Five _1"]))

# Missingness: share of cells that are NA among non-metadata columns
meta = {"TWIN_ID","StartDate","EndDate","Progress","Duration (in seconds)","Finished","RecordedDate"}
miss = []
for wave in range(1, 5):
    df = load_wave(wave, "numbers")
    cols = [c for c in df.columns if c not in meta]
    miss.append({"wave": wave, "n_item_cols": len(cols),
                 "cell_missing_rate": float(df[cols].isna().mean().mean()),
                 "cols_all_missing": int(df[cols].isna().all().sum())})
print("\nNumeric-file missingness (includes skip-logic / unassigned conditions):")
display(pd.DataFrame(miss))


First rows of each CSV (row0 = names, row2 ≈ ImportId):


,file,n_columns,row0,importid_counts
0,wave_1_labels_anonymized.csv,254,"[TWIN_ID, StartDate, EndDate, Progress, Duration (in seconds), Finished]","[0, 0, 253, 0]"
1,wave_1_numbers_anonymized.csv,254,"[TWIN_ID, StartDate, EndDate, Progress, Duration (in seconds), Finished]","[0, 0, 253, 0]"
2,wave_2_labels_anonymized.csv,424,"[TWIN_ID, StartDate, EndDate, Progress, Duration (in seconds), Finished]","[0, 0, 423, 0]"
3,wave_2_numbers_anonymized.csv,424,"[TWIN_ID, StartDate, EndDate, Progress, Duration (in seconds), Finished]","[0, 0, 423, 0]"
4,wave_3_labels_anonymized.csv,259,"[TWIN_ID, StartDate, EndDate, Progress, Duration (in seconds), Finished]","[0, 0, 258, 0]"
5,wave_3_numbers_anonymized.csv,259,"[TWIN_ID, StartDate, EndDate, Progress, Duration (in seconds), Finished]","[0, 0, 258, 0]"
6,wave_4_labels_anonymized.csv,164,"[TWIN_ID, StartDate, EndDate, Progress, Duration (in seconds), Finished]","[0, 0, 163, 0]"
7,wave_4_numbers_anonymized.csv,164,"[TWIN_ID, StartDate, EndDate, Progress, Duration (in seconds), Finished]","[0, 0, 163, 0]"


Clean-loaded participant tables:


,file,n_rows,n_cols,duplicate_TWIN_ID,missing_TWIN_ID,Finished=TRUE
0,wave_1_labels_anonymized.csv,2058,254,0,0,2058
1,wave_1_numbers_anonymized.csv,2058,254,0,0,0
2,wave_2_labels_anonymized.csv,2058,424,0,0,2058
3,wave_2_numbers_anonymized.csv,2058,424,0,0,0
4,wave_3_labels_anonymized.csv,2058,259,0,0,2058
5,wave_3_numbers_anonymized.csv,2058,259,0,0,0
6,wave_4_labels_anonymized.csv,2058,164,0,0,2058
7,wave_4_numbers_anonymized.csv,2058,164,0,0,0


Labels vs numbers share columns: True

Sex (Q12): labels vs numeric codes
Q12      1.0   2.0
Q12               
Female     0  1044
Male    1014     0

Big Five item 1: labels vs codes
Big Five _1                 1.0  2.0  3.0  4.0  5.0
Big Five _1                                        
Agree a little                0    0    0  631    0
Agree strongly                0    0    0    0  295
Disagree a little             0  503    0    0    0
Disagree strongly           413    0    0    0    0
Neither agree nor disagree    0    0  216    0    0



Numeric-file missingness (includes skip-logic / unassigned conditions):


,wave,n_item_cols,cell_missing_rate,cols_all_missing
0,1,247,0.021710,1
1,2,417,0.281800,3
2,3,252,0.318689,1
3,4,157,0.199869,1


## 4. Persona JSON structure

Persona files are a **list of blocks**. Each block has `Questions`. A JSON **question object** is not the same thing as a survey **item**: a Matrix object can hold many row-level responses. The paper’s “~500 questions” refers to **item-level** content across waves 1–3, not the count of JSON objects.

Observed `QuestionType` values in the local files: **MC, Matrix, TE, DB** in persona JSON; hold-out blocks additionally include **Slider**.


In [4]:
persona_stats = summarize_persona_files()
t1_stats = summarize_answer_block_files("t1")

print("Persona JSON (waves 1–3 non-holdout)")
display(persona_stats[["n_blocks","n_json_question_objects","n_non_db_objects","n_item_level_responses"]].describe().round(2))
print("Block-count frequencies:", persona_stats["n_blocks"].value_counts().to_dict())
print("JSON-object-count frequencies:", persona_stats["n_json_question_objects"].value_counts().to_dict())

type_cols = [c for c in persona_stats.columns if c.startswith("type_")]
print("\nQuestion-object type totals (persona, all participants):")
display(persona_stats[type_cols].sum().rename("n_objects").to_frame())

print("\nT1 hold-out answer blocks")
display(t1_stats[["n_blocks","n_json_question_objects","n_item_level_responses"]].describe().round(2))
print("T1 JSON-object counts:", t1_stats["n_json_question_objects"].value_counts().to_dict())

# Types in one T1 file (includes Slider)
qtypes_t1 = Counter()
for _, q in iter_questions(load_json(ANSWER_BLOCK_DIR / "pid_1_wave4_Q_wave1_3_A.json")):
    qtypes_t1[q.get("QuestionType")] += 1
print("T1 pid_1 question-object types:", dict(qtypes_t1))

print("\n--- Sanitized examples (pid_1 persona + T1) ---")
examples = [
    ("MC", PERSONA_DIR / "pid_1_mega_persona.json"),
    ("Matrix", PERSONA_DIR / "pid_1_mega_persona.json"),
    ("TE", ANSWER_BLOCK_DIR / "pid_1_wave4_Q_wave1_3_A.json"),
    ("Slider", ANSWER_BLOCK_DIR / "pid_1_wave4_Q_wave1_3_A.json"),
    ("DB", PERSONA_DIR / "pid_1_mega_persona.json"),
]
for qtype, path in examples:
    q = find_example_question(path, qtype, require_answers=(qtype != "DB"))
    print(f"\n### {qtype}  ({path.name})")
    print(json.dumps(sanitize_question_example(q), indent=2)[:1800])


Persona JSON (waves 1–3 non-holdout)


,n_blocks,n_json_question_objects,n_non_db_objects,n_item_level_responses
count,2058.00,2058.00,2058.00,2058.00
mean,13.00,171.98,157.98,536.98
std,0.05,0.17,0.17,0.17
min,12.00,170.00,156.00,535.00
25%,13.00,172.00,158.00,537.00
50%,13.00,172.00,158.00,537.00
75%,13.00,172.00,158.00,537.00
max,13.00,172.00,158.00,537.00


Block-count frequencies: {13: 2052, 12: 6}
JSON-object-count frequencies: {172: 2019, 171: 31, 170: 8}

Question-object type totals (persona, all participants):


,n_objects
type_MC,220199
type_DB,28812
type_Matrix,59682
type_TE,45236



T1 hold-out answer blocks


,n_blocks,n_json_question_objects,n_item_level_responses
count,2058.0,2058.0,2058.00
mean,18.0,64.0,96.01
std,0.0,0.0,2.00
min,18.0,64.0,94.00
25%,18.0,64.0,94.00
50%,18.0,64.0,98.00
75%,18.0,64.0,98.00
max,18.0,64.0,98.00


T1 JSON-object counts: {64: 2058}
T1 pid_1 question-object types: {'Matrix': 5, 'Slider': 2, 'MC': 53, 'TE': 3, 'DB': 1}

--- Sanitized examples (pid_1 persona + T1) ---

### MC  (pid_1_mega_persona.json)
{
  "QuestionID": "QID11",
  "QuestionType": "MC",
  "QuestionText": "Which part of the United States do you currently live in?",
  "Settings.Selector": "SAVR",
  "n_Options": 5,
  "n_Rows": 0,
  "n_Columns": 0,
  "n_Statements": 0,
  "AnswerKeys": [
    "SelectedByPosition",
    "SelectedText"
  ],
  "AnswersPreview": {
    "SelectedByPosition": 2,
    "SelectedText": "Midwest (ND, SD, NE, KS, MN, IA, MO, WI, IL, MI, IN, OH)"
  }
}

### Matrix  (pid_1_mega_persona.json)
{
  "QuestionID": "QID25",
  "QuestionType": "Matrix",
  "QuestionText": "Here are a number of characteristics that may or may not apply to you. Please indicate next to each statement the extent to which you agree or disagree with that statement. I see myself as someone who...",
  "Settings.Selector": "Likert",
  "n_O

**How answers are stored (response-bearing types):**

| Type | What it is | Answer representation |
|---|---|---|
| **MC** | Single or multi choice | `SelectedByPosition` (1-indexed option number) and `SelectedText` |
| **Matrix** | Grid; one object, many rows | Parallel lists `SelectedByPosition` / `SelectedText`, one entry per row |
| **TE** | Text / numeric entry | `Answers.Text` (anchoring and sunk-cost numbers are TE with `ContentType: ValidNumber`) |
| **Slider** | Bounded numeric | `Answers.Values` (list; one value per statement) |
| **DB** | Instructional copy | No answers |

A typical persona has **13 blocks**, **~172 JSON question objects** (~158 non-DB), and **~537 item-level responses**. That last number is the right comparison to “500 questions,” not 172.


## 5. Evaluation tasks and scoring

Hold-out scoring is defined in `evaluation/mad_accuracy_evaluation.py` (column ranges, 17-task map, anchoring deciles). We **reimplement** it in `src/evaluation_utils.py` and extract T1/T2 from answer-block JSON rather than calling the paper’s evaluation script (that script also expects LLM output we do not use here).

**Which tasks are scored.** Seventeen tasks covering the heuristics/biases batteries plus the 40-item pricing study. Between-subject experiments contribute **one variant per person** (the condition they were assigned in waves 1–3 and again in Wave 4). Within-subject batteries (false consensus, non-separability, etc.) contribute several items each.

**Item accuracy** (paper + repo):

\[
\mathrm{accuracy} = 1 - \frac{|\hat{y} - y|}{y_{\max} - y_{\min}}
\]

- Binary items have range \(1\) (e.g. codes 1–2), so this is exact match.
- Ordinal Likert items use the option span (e.g. 1–5 → range 4).
- **Anchoring** free-response numbers are unbounded. The repo converts T1 values to **deciles** (cutpoints = T1 percentiles 10,20,…,90), maps both T1 and T2 into \{1,…,10\}, then uses range \(9\). The paper text says “based on wave 2”; T1 *is* that original wave-2 administration.

**Aggregation (why task-level means matter):**

1. Average item accuracies **within a task for one person** (pricing’s 40 binary items would otherwise swamp omission’s single item).
2. Average people within a task → one accuracy per task (with a t-interval).
3. **Overall = unweighted mean of the 17 task means.**

Because every completer has all 17 tasks, that overall number equals the mean of person-level (equal-task-weight) accuracies.


In [5]:
catalog = task_catalog()
display(catalog[["task","response_type","n_scored_columns","scoring_method"]])
print("Total mapped columns (includes mutually exclusive between-subject variants):",
      catalog["n_scored_columns"].sum())
print("Number of tasks:", len(catalog))

# Worked examples of the accuracy formula
print("\nWorked examples")
print("  Binary: T1=1, T2=1, range 1–2 →", round(item_accuracy(1, 1, 1, 2), 3))
print("  Binary: T1=1, T2=2, range 1–2 →", round(item_accuracy(1, 2, 1, 2), 3))
print("  Likert: T1=3, T2=5, range 1–5 → 1 - |3-5|/4 =", round(item_accuracy(3, 5, 1, 5), 3))
print("  Slider: T1=70, T2=55, range 0–100 →", round(item_accuracy(70, 55, 0, 100), 3))

# Tiny anchoring decile illustration from real T1 Q166
t1_preview, t2_preview, _ = load_holdout_matrices()
vals = t1_preview["Q166"].dropna()
th = np.percentile(vals, np.arange(10, 100, 10))
print("\nQ166 T1 decile cutpoints (first 3 / last 3):", np.round(th[:3], 2), "…", np.round(th[-3:], 2))
print("  raw 10 → decile", assign_decile(10, th), "; raw 50 →", assign_decile(50, th), "; raw 5000 →", assign_decile(5000, th))


,task,response_type,n_scored_columns,scoring_method
0,Allais,binary / 2-option MC,2,Exact match (binary); accuracy 0/1
1,WTA/WTP-Thaler,ordinal / bounded numeric,3,1 - |pred-truth| / (10-1)
2,absolute vs. relative savings,binary / 2-option MC,2,Exact match (binary); accuracy 0/1
3,anchoring and adjustment,unbounded numeric (TE),4,"Decile-bin unbounded estimates, then 1 - |d|/9"
4,base rate,ordinal Likert + 0–100 slider,2,1 - |pred-truth| / range (Likert 1–5 and/or 0–100)
5,conjunction problem (Linda),ordinal / bounded numeric,6,1 - |pred-truth| / (6-1)
6,denominator neglect,binary / 2-option MC,1,Exact match (binary); accuracy 0/1
7,false consensus,ordinal Likert + 0–100 slider,20,1 - |pred-truth| / range (Likert 1–5 and/or 0–100)
8,framing problem,ordinal / bounded numeric,2,1 - |pred-truth| / (6-1)
9,less is more,ordinal / bounded numeric,9,1 - |pred-truth| / (6-1)


Total mapped columns (includes mutually exclusive between-subject variants): 122
Number of tasks: 17

Worked examples
  Binary: T1=1, T2=1, range 1–2 → 1.0
  Binary: T1=1, T2=2, range 1–2 → 0.0
  Likert: T1=3, T2=5, range 1–5 → 1 - |3-5|/4 = 0.5
  Slider: T1=70, T2=55, range 0–100 → 0.85



Q166 T1 decile cutpoints (first 3 / last 3): [15. 25. 32.] … [50. 54. 70.]
  raw 10 → decile 1.0 ; raw 50 → 6.0 ; raw 5000 → 10.0


## 6. Participant distributions and representativeness

Demographics are taken from **Wave 1 labels** (`Q11`–`Q24`). The paper (Toubia et al.) states Wave 1 targeted **2,500 representative US respondents on Prolific, sampled by age, sex, and ethnicity**; 2,509 completed Wave 1; **2,058 completed all four waves** and constitute this release.

Language used below is deliberate:

- This is **quota / targeted representativeness** on a few demographics, not a demonstration that the sample matches the US joint distribution on education, income, politics, etc.
- The analytic file is **study completers**. People who dropped after Wave 1 are **not in `data/`**, so attrition bias cannot be estimated from the release alone.


In [6]:
demo = load_demographics()
assert len(demo) == 2058

def count_pct(series):
    vc = series.value_counts(dropna=False)
    return pd.DataFrame({"count": vc, "percent": (100 * vc / len(series)).round(1)})

fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.0))

age_order = ["18-29", "30-49", "50-64", "65+"]
age = demo["age"].value_counts().reindex(age_order)
axes[0].barh(age_order, 100 * age / len(demo), color="#4C78A8")
axes[0].set_xlabel("Percent of sample")
axes[0].set_title("Age (Wave 1)")
for i, (n, p) in enumerate(zip(age, 100 * age / len(demo))):
    axes[0].text(p + 0.4, i, f"{n} ({p:.1f}%)", va="center", fontsize=9)

sex = demo["sex"].value_counts()
axes[1].bar(sex.index.astype(str), 100 * sex / len(demo), color="#4C78A8", width=0.55)
axes[1].set_ylabel("Percent of sample")
axes[1].set_title("Sex assigned at birth (Wave 1)")
for x, n, p in zip(sex.index, sex, 100 * sex / len(demo)):
    axes[1].text(x, p + 0.8, f"{n}\n({p:.1f}%)", ha="center", va="bottom", fontsize=9)
axes[1].set_ylim(0, 65)

race_order = ["White", "Black", "Hispanic", "Asian", "Other"]
race = demo["race"].value_counts().reindex(race_order)
axes[2].barh(race_order[::-1], (100 * race / len(demo)).reindex(race_order[::-1]), color="#4C78A8")
axes[2].set_xlabel("Percent of sample")
axes[2].set_title("Race / origin (Wave 1)")
for i, lab in enumerate(race_order[::-1]):
    n, p = race[lab], 100 * race[lab] / len(demo)
    axes[2].text(p + 0.4, i, f"{n} ({p:.1f}%)", va="center", fontsize=9)

fig.tight_layout()
fig.savefig(FIG_DIR / "demographics_age_sex_race.png", bbox_inches="tight")
plt.show()

other_cols = [
    "region", "education", "income", "employment",
    "political_party", "political_ideology", "marital_status",
    "household_size", "us_citizen", "religious_attendance",
]
rows = []
for col in other_cols:
    for val, n in demo[col].value_counts(dropna=False).items():
        rows.append({"variable": col, "value": val, "count": int(n), "percent": round(100 * n / len(demo), 1)})
other_tbl = pd.DataFrame(rows)
print("Other demographics (counts and % of 2,058 completers)")
display(other_tbl)
other_tbl.to_csv(FIG_DIR / "demographics_other.csv", index=False)


Other demographics (counts and % of 2,058 completers)


/tmp/ipykernel_93209/115908513.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,variable,value,count,percent
0,region,"South (TX, OK, AR, LA, KY, TN, MS, AL, WV, DC, MD, DE, VA, NC, SC, GA, FL)",834,40.5
1,region,"West (WA, OR, ID, MT, WY, CA, NV, UT, CO, AZ, NM)",494,24.0
2,region,"Midwest (ND, SD, NE, KS, MN, IA, MO, WI, IL, MI, IN, OH)",372,18.1
3,region,"Northeast (PA, NY, NJ, RI, CT, MA, VT, NH, ME)",342,16.6
4,region,"Pacific (HI, AK)",16,0.8
5,education,College graduate/some postgrad,735,35.7
6,education,"Some college, no degree",468,22.7
7,education,Postgraduate,313,15.2
8,education,High school graduate,272,13.2
9,education,Associate's degree,253,12.3


## 7. Human test–retest reliability (main result)

Inputs: `wave4_Q_wave1_3_A` (T1) and `wave4_Q_wave4_A` (T2). We first confirm that each pair is the **same question/condition structure**, then score with the protocol in Section 5.

Test–retest is a **human reliability benchmark** / the paper’s **practical ceiling**. It is not a strict mathematical upper bound: a model could in principle be more stable than a person, and some T1–T2 disagreements are noise rather than “true” change.


In [7]:
struct = compare_all_t1_t2_structure()
print("T1/T2 pairs:", len(struct))
print("Same number of question objects:", bool(struct["same_length"].all()))
print("Identical structure and order (excluding Answers):", bool(struct["same_order"].all()))
print("Max mismatched objects among equal-length pairs:", struct["n_mismatch"].max())

print("\nExtracting T1/T2 numeric matrices and scoring (all 2,058 people)…")
t1, t2, import_map = load_holdout_matrices()
print("T1/T2 shapes:", t1.shape, t2.shape)
print("Mean scored items per person with both T1 and T2:",
      float((t1.notna() & t2.notna()).sum(axis=1).mean()))

t1_dec, t2_dec = apply_anchoring_deciles(t1, t2)
tt = compute_test_retest(t1_dec, t2_dec)

overall = tt["overall_equal_task_mean"]
person_ci = tt["overall_from_persons"]
print("\n=== HEADLINE TEST–RETEST ===")
print(f"Equal-task-weight overall accuracy: {overall*100:.4f}%")
print(f"Person-level mean (same, because all 17 tasks present): {person_ci['mean']*100:.4f}%")
print(f"Person-level 95% CI: {person_ci['ci_low']*100:.4f}–{person_ci['ci_high']*100:.4f}%  (n={person_ci['n']})")
print("Paper reports 81.72%. Absolute difference: "
      f"{abs(overall*100 - 81.72):.4f} pp (validation check, not our source).")

task_df = tt["task_df"].copy()
task_df["accuracy_pct"] = task_df["accuracy"] * 100
task_df["ci_low_pct"] = task_df["ci_low"] * 100
task_df["ci_high_pct"] = task_df["ci_high"] * 100
display(task_df[["task","n_respondents","accuracy_pct","ci_low_pct","ci_high_pct"]].round(2))


T1/T2 pairs: 2058
Same number of question objects: True
Identical structure and order (excluding Answers): True
Max mismatched objects among equal-length pairs: 0

Extracting T1/T2 numeric matrices and scoring (all 2,058 people)…


T1/T2 shapes: (2058, 122) (2058, 122)
Mean scored items per person with both T1 and T2: 94.00583090379008



=== HEADLINE TEST–RETEST ===
Equal-task-weight overall accuracy: 81.7317%
Person-level mean (same, because all 17 tasks present): 81.7317%
Person-level 95% CI: 81.4223–82.0410%  (n=2058)
Paper reports 81.72%. Absolute difference: 0.0117 pp (validation check, not our source).


,task,n_respondents,accuracy_pct,ci_low_pct,ci_high_pct
0,denominator neglect,2058,72.16,70.22,74.10
1,Allais,2058,74.05,72.16,75.95
2,prob matching vs. max,2058,74.76,73.70,75.82
3,sunk cost fallacy,2058,78.84,77.79,79.89
4,absolute vs. relative savings,2058,79.15,77.40,80.91
5,framing problem,2058,80.27,79.35,81.19
6,omission,2058,80.58,79.55,81.61
7,conjunction problem (Linda),2058,82.41,81.84,82.97
8,less is more,2058,83.15,82.58,83.71
9,base rate,2058,83.46,82.61,84.30


In [8]:
# Figure: human test-retest by task
plot_df = task_df.sort_values("accuracy")
y = np.arange(len(plot_df))
xerr = np.vstack([
    plot_df["accuracy_pct"] - plot_df["ci_low_pct"],
    plot_df["ci_high_pct"] - plot_df["accuracy_pct"],
])

fig, ax = plt.subplots(figsize=(9.5, 6.8))
ax.barh(y, plot_df["accuracy_pct"], color="#4C78A8", height=0.7,
        xerr=xerr, capsize=3, ecolor="#333333", error_kw={"linewidth": 0.8})
ax.axvline(overall * 100, color="#C44E52", ls="--", lw=1.4,
           label=f"Overall (equal task weight) = {overall*100:.2f}%")
ax.set_yticks(y)
ax.set_yticklabels(plot_df["task"])
ax.set_xlabel("Test–retest accuracy (%)")
ax.set_xlim(65, 95)
ax.set_title("Human test–retest accuracy by task (T1 vs T2)")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / "test_retest_by_task.png", bbox_inches="tight")
plt.show()


/tmp/ipykernel_93209/2172187044.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
person = tt["person_df"]["accuracy"] * 100
qs = person.quantile([0.05, 0.25, 0.5, 0.75, 0.95])
print("Participant-level test–retest accuracy (%)")
print(f"  n      {len(person)}")
print(f"  mean   {person.mean():.2f}")
print(f"  median {person.median():.2f}")
print(f"  sd     {person.std():.2f}")
print(f"  5th    {qs.loc[0.05]:.2f}")
print(f"  25th   {qs.loc[0.25]:.2f}")
print(f"  75th   {qs.loc[0.75]:.2f}")
print(f"  95th   {qs.loc[0.95]:.2f}")

fig, ax = plt.subplots(figsize=(8.2, 4.2))
ax.hist(person, bins=25, color="#4C78A8", edgecolor="white")
ax.axvline(person.mean(), color="#C44E52", ls="--", label=f"Mean {person.mean():.1f}%")
ax.axvline(person.median(), color="#55A868", ls=":", label=f"Median {person.median():.1f}%")
ax.set_xlabel("Person-level test–retest accuracy (%)")
ax.set_ylabel("Number of participants")
ax.set_title("Distribution of participant-level test–retest accuracy")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "test_retest_person_hist.png", bbox_inches="tight")
plt.show()


Participant-level test–retest accuracy (%)
  n      2058
  mean   81.73
  median 82.49
  sd     7.16
  5th    68.74
  25th   77.40
  75th   86.97
  95th   92.12


/tmp/ipykernel_93209/1613679179.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation.** People are not perfectly self-consistent over a ~2–4 week gap. A later model that hits 80% item accuracy is close to **human reliability**, not close to failure relative to 100%. Tasks with unstable human answers (denominator neglect, Allais, probability matching) will look “hard” for any predictor; high-reliability tasks (WTA/WTP, outcome bias) are easier ceilings. Report **per-task** numbers, not only the 81.7% headline.


## 8. Data quality checks


In [10]:
# Out-of-range: apply published bounds only AFTER anchoring deciles
ranges = get_column_ranges()
oor_rows = []
for col, (lo, hi) in ranges.items():
    if col not in t1_dec.columns:
        continue
    s = t1_dec[col].dropna()
    n = int(((s < lo) | (s > hi)).sum())
    if n:
        oor_rows.append({"column": col, "n_T1_out_of_range": n, "min": s.min(), "max": s.max(), "bounds": (lo, hi)})
oor = pd.DataFrame(oor_rows)

# Raw anchoring (before decile) is expected to fall outside 1–10
raw_anchor_oob = []
for col in ANCHOR_GROUP_A + ANCHOR_GROUP_B:
    if col in t1.columns:
        s = t1[col].dropna()
        raw_anchor_oob.append({"column": col, "n": int(s.notna().sum()), "min": s.min(), "max": s.max()})

# Missing scored items: between-subject variants are missing by design
both = t1.notna() & t2.notna()
per_person_items = both.sum(axis=1)

# Summaries
sum_lens = [p.read_text(encoding="utf-8").strip().__len__() for p in SUMMARY_DIR.glob("*.txt")]

quality = pd.DataFrame([
    {"check": "Duplicate participant IDs in any wave CSV",
     "result": "None (0 duplicates, 0 missing TWIN_ID, 2058 rows × 4 waves)",
     "implication": "Released tables are already restricted to completers"},
    {"check": "PID alignment across persona / T1 / T2 / summaries / Wave-1 CSV",
     "result": "Identical set {1,…,2058}",
     "implication": "No unmatched twins in local files"},
    {"check": "T1 vs T2 question fingerprints",
     "result": f"{int(struct['same_order'].sum())}/{len(struct)} pairs identical in ID/type/options/order",
     "implication": "Condition assignment was held fixed; T1/T2 differ in Answers only"},
    {"check": "Persona block / object count variation",
     "result": f"{int((persona_stats.n_blocks==13).sum())} people with 13 blocks; "
               f"{int((persona_stats.n_blocks==12).sum())} with 12. JSON objects 170–172. "
               f"Item-level responses 535–537 (mean {persona_stats.n_item_level_responses.mean():.2f})",
     "implication": "Small skip-logic or missing-block variation; not a broken schema"},
    {"check": "Empty persona summaries",
     "result": f"{sum(L==0 for L in sum_lens)} empty; length min={min(sum_lens)}, max={max(sum_lens)}",
     "implication": "full_persona summaries are present for every pid"},
    {"check": "T1/T2 missingness on scored columns",
     "result": f"Mean items with both answers: {per_person_items.mean():.2f} "
               f"(min {int(per_person_items.min())}, max {int(per_person_items.max())})",
     "implication": "Between-subject tasks contribute one variant; matching problem 1 vs 2 changes item count"},
    {"check": "Out-of-range vs post-decile bounds",
     "result": ("None after decile transform" if oor.empty else f"{len(oor)} columns still OOR"),
     "implication": "Anchoring raw values are unbounded by design; scoring uses deciles"},
    {"check": "Item-level accuracy outside [0,1]",
     "result": f"min={tt['item_df'].accuracy.min():.3f}, max={tt['item_df'].accuracy.max():.3f}",
     "implication": "No inverted or exploding scores under the published ranges"},
])
display(quality)
print("Raw anchoring numeric range (before deciles) — expected to be unbounded:")
display(pd.DataFrame(raw_anchor_oob))


,check,result,implication
0,Duplicate participant IDs in any wave CSV,"None (0 duplicates, 0 missing TWIN_ID, 2058 rows × 4 waves)",Released tables are already restricted to completers
1,PID alignment across persona / T1 / T2 / summaries / Wave-1 CSV,"Identical set {1,…,2058}",No unmatched twins in local files
2,T1 vs T2 question fingerprints,2058/2058 pairs identical in ID/type/options/order,Condition assignment was held fixed; T1/T2 differ in Answers only
3,Persona block / object count variation,2052 people with 13 blocks; 6 with 12. JSON objects 170–172. Item-level resp...,Small skip-logic or missing-block variation; not a broken schema
4,Empty persona summaries,"0 empty; length min=11602, max=18482",full_persona summaries are present for every pid
5,T1/T2 missingness on scored columns,"Mean items with both answers: 94.01 (min 92, max 96)",Between-subject tasks contribute one variant; matching problem 1 vs 2 change...
6,Out-of-range vs post-decile bounds,None after decile transform,Anchoring raw values are unbounded by design; scoring uses deciles
7,"Item-level accuracy outside [0,1]","min=0.000, max=1.000",No inverted or exploding scores under the published ranges


Raw anchoring numeric range (before deciles) — expected to be unbounded:


,column,n,min,max
0,Q164,1002,0.0,193.0
1,Q166,1056,0.0,5000.0
2,Q168,1049,2.0,2200.0
3,Q170,1009,0.0,11000.0


## 9. Biases and limitations

**(a) Directly observed in these files**

- US-centric items (US region, citizenship; political stimuli). **2,054 / 2,058** report US citizenship.
- Completer-only tables: every wave CSV has 2,058 finished rows. Wave 1 dropouts are not here.
- Coarse bins (4 age groups, 5 race codes, 5 income bands).
- Hold-out tasks are **repeats**, ~1–4 weeks later (Wave 1 launched 2025-01-29, Wave 4 2025-02-25 per paper dates; timestamps in the CSVs are consistent with that window). Memory, learning, and familiarity can inflate T1–T2 agreement.
- Human reliability varies by task (Section 7). Overall 81.7% hides ~72–89% spread.

**(b) Documented by the paper / repo, not independently re-derived as a sampling design**

- Prolific; Wave 1 **targeted** representativeness on age, sex, ethnicity (quota sampling), not a full probability sample.
- Original funnel: 2,509 → 2,263 → 2,252 → 2,058. This release does not include the non-completers, so we **cannot** quantify attrition bias from `data/` alone.
- Survey answers ≠ incentivized real-world behavior (pricing is hypothetical purchase; WTA/WTP is a vignette).
- Coverage is social-science / behavioral-econ / a 40-product pricing study — not an all-domain model of a person.
- Paper notes political and domain-specific views as a place digital twins may struggle; we did not re-analyze that here.

**(c) Reasonable implications**

- Longer gaps would likely **lower** test–retest; this ceiling is tied to a short panel.
- Later models must keep T1 hold-out answers **out** of the persona. Using `wave1_3_persona_json` (as shipped) is the evaluation-safe split; concatenating T1 into the prompt would leak the target.


## 10. Implications for later modeling

1. **Evaluation data have intrinsic human temporal variability.** Interpret model accuracy against ~82% test–retest, not 100%.
2. **Normalize/contextualize overall performance** as a ratio to human reliability (the paper’s “relative accuracy”), and keep a random-choice baseline for scale.
3. **Always report per-task performance.** Reliability (and later, model skill) moves a lot across the 17 tasks.
4. **Preserve the evaluation-safe persona.** Waves 1–3 non-holdout JSON / summaries are inputs; T1 answers are labels for training-set-style prediction of T2 or of T1 itself — they must not be mixed into the prompt by accident.
5. **Score with the published MAD rule**, including anchoring deciles. Do not treat every numeric code as a continuous quantity with range = observed max−min.
6. **Persona records are long and nested** (~537 items, ~13k-character summaries). Encoding choices (full text vs summary vs tabular features) will matter; that is a later deliverable.

This notebook stops here. No model training, fine-tuning, or application design.


In [11]:
print("=" * 60)
print("RECAP (computed this run)")
print("=" * 60)
print(f"Participants: {tt['n_participants']}")
print(f"Scored eval columns in matrix: {tt['n_scored_columns']}")
print(f"Overall human test–retest: {overall*100:.2f}%")
print(f"Person-level 95% CI: {person_ci['ci_low']*100:.2f}–{person_ci['ci_high']*100:.2f}%")
print("Task accuracy range: "
      f"{task_df.accuracy_pct.min():.1f}% ({task_df.loc[task_df.accuracy.idxmin(),'task']}) to "
      f"{task_df.accuracy_pct.max():.1f}% ({task_df.loc[task_df.accuracy.idxmax(),'task']})")
print("Figures written to:", FIG_DIR)
for p in sorted(FIG_DIR.glob("*")):
    print(" ", p.relative_to(ROOT))


RECAP (computed this run)
Participants: 2058
Scored eval columns in matrix: 122
Overall human test–retest: 81.73%
Person-level 95% CI: 81.42–82.04%
Task accuracy range: 72.2% (denominator neglect) to 89.2% (WTA/WTP-Thaler)
Figures written to: /home/user/PycharmProjects/Twin-2K-500-Assignment/Digital-Twin-Simulation/reports/figures
  reports/figures/demographics_age_sex_race.png
  reports/figures/demographics_other.csv
  reports/figures/test_retest_by_task.png
  reports/figures/test_retest_person_hist.png
